# Pertemuan 7 — Praktikum 4: Perbandingan Annoy, FAISS, dan HNSW

Praktikum ini membandingkan ketiga pustaka ANN pada dataset yang sama. Data yang dipakai berjumlah satu juta vektor berdimensi lima. Ketiga pustaka dibangun pada data tersebut, kemudian satu query dikirim ke masing-masing index.

Perbandingan mencakup dua hal, yaitu waktu pembuatan index (*build time*) dan waktu pencarian (*query time*). Pemisahan kedua ukuran ini penting karena pembuatan index dilakukan sekali di awal, sedangkan pencarian dilakukan berulang.

Sumber: [Praktikum 4](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js06-ann-approximate-nearest-neighbors/praktikum-4).

## 1. Menyiapkan data

Sel berikut membuat satu juta vektor acak berdimensi lima beserta satu vektor query, kemudian memuat ketiga pustaka ANN.

In [1]:
%matplotlib inline

import time

import faiss
import hnswlib
import numpy as np
import pandas as pd
from annoy import AnnoyIndex

In [2]:
# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000  # bisa coba 100_000 dulu jika RAM terbatas
dim = 5

np.random.seed(42)
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

## 2. Annoy

Annoy dibangun dengan 10 pohon. Pada tahap pencarian, nilai `search_k` ditetapkan secara eksplisit. Nilai bawaan Annoy adalah `n_trees × jumlah data`, yang pada satu juta data berarti sepuluh juta kandidat untuk setiap query, sehingga pencariannya menjadi tidak praktis.

In [3]:
# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, search_k=10_000, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

=== Annoy ===


Build time: 5.133230924606323 detik
Query time: 0.0015256404876708984 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...


## 3. FAISS

FAISS memakai `IndexFlatL2`, yaitu index yang menghitung jarak terhadap seluruh vektor tanpa kompresi. Cara ini bersifat exact, sehingga hasilnya dapat dipakai sebagai acuan kebenaran bagi kedua pustaka lainnya.

In [4]:
# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time_faiss = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time_faiss = time.time() - start

print("Build time:", build_time_faiss, "detik")
print("Query time:", query_time_faiss, "detik")
print("Neighbors:", indices[0][:5], "...")

=== FAISS (IndexFlatL2) ===
Build time: 0.0034568309783935547 detik
Query time: 0.016913175582885742 detik
Neighbors: [901095 495168 561017 163986 185734] ...


## 4. HNSW

HNSW dibangun dengan `M = 16` dan `ef_construction = 200`, kemudian pencarian memakai `ef = 50`.

In [5]:
# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time_hnsw = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances_hnsw = hnsw_index.knn_query(query, k=k)
query_time_hnsw = time.time() - start

print("Build time:", build_time_hnsw, "detik")
print("Query time:", query_time_hnsw, "detik")
print("Neighbors:", labels[0][:5], "...")

=== HNSW (hnswlib) ===


Build time: 25.00200891494751 detik
Query time: 0.003854036331176758 detik
Neighbors: [901095 495168 561017 163986 185734] ...


## 5. Ringkasan perbandingan

Sel berikut menyusun waktu pembuatan index dan waktu pencarian ketiga pustaka ke dalam satu tabel agar mudah dibandingkan.

In [6]:
ringkasan = pd.DataFrame([
    {"Library": "Annoy", "Build time (detik)": round(build_time, 4), "Query time (detik)": round(query_time, 6),
     "Neighbors (5 pertama)": str(neighbors[0][:5])},
    {"Library": "FAISS (IndexFlatL2)", "Build time (detik)": round(build_time_faiss, 4),
     "Query time (detik)": round(query_time_faiss, 6), "Neighbors (5 pertama)": str(list(indices[0][:5]))},
    {"Library": "HNSW (hnswlib)", "Build time (detik)": round(build_time_hnsw, 4),
     "Query time (detik)": round(query_time_hnsw, 6), "Neighbors (5 pertama)": str(list(labels[0][:5]))},
])

display(ringkasan)

,Library,Build time (detik),Query time (detik),Neighbors (5 pertama)
0,Annoy,5.1332,0.001526,"[901095, 495168, 561017, 163986, 185734]"
1,FAISS (IndexFlatL2),0.0035,0.016913,"[np.int64(901095), np.int64(495168), np.int64(..."
2,HNSW (hnswlib),25.0020,0.003854,"[np.uint64(901095), np.uint64(495168), np.uint..."


## 6. Pengaruh metrik jarak

Percobaan diulang dengan metrik jarak yang berbeda. Pada Annoy dipakai metrik Euclidean dan Angular, pada FAISS dipakai L2 dan Inner Product, dan pada HNSW dipakai `l2` serta `ip`. Ukuran data dan dimensi dibuat sama seperti sebelumnya agar hasilnya sebanding.

In [7]:
def uji_tiga_pustaka(n_data, dim, metrik, k=10, seed=42):
    """Mengukur build time dan query time ketiga pustaka untuk satu metrik."""
    rng = np.random.default_rng(seed)
    data = rng.random((n_data, dim)).astype(np.float32)
    titik = rng.random((1, dim)).astype(np.float32)

    metric_annoy = 'euclidean' if metrik in ('Euclidean', 'L2') else 'angular'

    # Annoy
    ann = AnnoyIndex(dim, metric_annoy)
    t0 = time.perf_counter()
    for i in range(n_data):
        ann.add_item(i, data[i])
    ann.build(10)
    build_annoy = time.perf_counter() - t0
    t0 = time.perf_counter()
    ann.get_nns_by_vector(titik[0], k, search_k=10_000)
    query_annoy = (time.perf_counter() - t0) * 1000

    # FAISS
    ruang_faiss = faiss.IndexFlatL2(dim) if metrik in ('Euclidean', 'L2') else faiss.IndexFlatIP(dim)
    t0 = time.perf_counter()
    ruang_faiss.add(data)
    build_faiss = time.perf_counter() - t0
    t0 = time.perf_counter()
    ruang_faiss.search(titik, k)
    query_faiss = (time.perf_counter() - t0) * 1000

    # HNSW
    space = 'l2' if metrik in ('Euclidean', 'L2') else 'ip'
    h = hnswlib.Index(space=space, dim=dim)
    t0 = time.perf_counter()
    h.init_index(max_elements=n_data, ef_construction=200, M=16)
    h.add_items(data)
    build_hnsw = time.perf_counter() - t0
    h.set_ef(50)
    t0 = time.perf_counter()
    h.knn_query(titik, k=k)
    query_hnsw = (time.perf_counter() - t0) * 1000

    return build_annoy, query_annoy, build_faiss, query_faiss, build_hnsw, query_hnsw


baris = []
for metrik in ('Euclidean / L2', 'Angular / Inner Product'):
    b_ann, q_ann, b_fai, q_fai, b_hn, q_hn = uji_tiga_pustaka(1_000_000, 5, metrik)
    baris.append({
        "Metrik": metrik,
        "Build Annoy (s)": round(b_ann, 3),
        "Query Annoy (ms)": round(q_ann, 4),
        "Build FAISS (s)": round(b_fai, 3),
        "Query FAISS (ms)": round(q_fai, 4),
        "Build HNSW (s)": round(b_hn, 3),
        "Query HNSW (ms)": round(q_hn, 4),
    })

display(pd.DataFrame(baris))

,Metrik,Build Annoy (s),Query Annoy (ms),Build FAISS (s),Query FAISS (ms),Build HNSW (s),Query HNSW (ms)
0,Euclidean / L2,7.180,1.0950,0.004,8.4966,19.994,0.1041
1,Angular / Inner Product,6.856,1.1493,0.003,0.7780,20.303,0.0875


## Kesimpulan

Ketiga pustaka menghasilkan tetangga terdekat yang berdekatan pada dataset satu juta vektor berdimensi lima, tetapi dengan karakteristik waktu yang berbeda. FAISS `IndexFlatL2` memiliki waktu pembuatan index paling singkat karena hanya menyimpan vektor tanpa struktur tambahan, namun waktu pencariannya paling lama karena menghitung jarak ke seluruh data. Annoy dan HNSW membangun struktur tambahan sehingga waktu pembuatan indexnya lebih lama, tetapi waktu pencariannya jauh lebih singkat. HNSW memiliki waktu pencarian terkecil pada percobaan ini, sedangkan Annoy unggul pada kesederhanaan dan waktu pembuatan index yang lebih singkat.